# VNetra Resume Training (Google Colab)

Notebook ini khusus digunakan untuk **melanjutkan (resume)** proses pelatihan model YOLO11 yang terhenti di Google Colab. 
Untuk menghindari error limitasi I/O dari Google Drive dan mempercepat proses training, notebook ini akan:
1. Mengekstrak Dataset Master dari Google Drive ke memori lokal Colab.
2. Mengekstrak Checkpoint Model (`last.pt`) dan history training (`results.csv`) dari Google Drive ke memori lokal Colab.
3. Melakukan Resume Training di memori lokal.
4. Menyimpan ulang (ZIP) hasil training yang baru ke Google Drive setelah selesai.

## 1. Setup Environment & Mount Google Drive
Langkah ini akan menyambungkan Colab ke Google Drive Anda dan menginstal *library* yang dibutuhkan (Ultralytics, Roboflow, dll).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# --- KONFIGURASI EKSPERIMEN (BISA DIUBAH) ---
EXPERIMENT_ID = 1

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install ultralytics roboflow pyyaml fiftyone Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

print("Environment siap!")

## 2. Ekstrak Dataset dan Model ke Penyimpanan Runtime (Lokal)
Bagian ini sangat krusial. Kita tidak boleh me-resume training dengan membaca file `.pt` langsung dari Google Drive karena akan menyebabkan error lambatnya baca-tulis (I/O Throttle). Kita akan mengekstrak file master dataset dan ZIP model ke memori lokal Colab (`/content/`).

In [ ]:
import zipfile
import os
import shutil

print("1. Mengekstrak Dataset Master ke Runtime...")
master_zip = f'{INPUT_DIR}/vnetra_master_dataset.zip'
master_dir = '/content/vnetra_master_dataset'

if not os.path.exists(master_dir):
    if os.path.exists(master_zip):
        shutil.unpack_archive(master_zip, '/content/vnetra_master_dataset')
        print(" Dataset berhasil diekstrak ke lokal!")
    else:
        print(f" ERROR: {master_zip} tidak ditemukan!")
else:
    print(" Dataset lokal sudah ada.")

print("\n2. Mengekstrak Model (Checkpoint) ke Runtime...")
model_dir = '/content/runs/detect/vnetra_training'

# Memakai hasil zip dari train notebook
training_zip = f'{OUTPUT_DIR}/vnetra_training_results.zip'

if not os.path.exists(model_dir):
    if os.path.exists(training_zip):
        print(f" Mengekstrak {training_zip}...")
        os.makedirs(model_dir, exist_ok=True)
        shutil.unpack_archive(training_zip, model_dir)
        print(" Model berhasil diekstrak ke lokal!")
    else:
        print(f" ERROR: Tidak menemukan file zip hasil training di {OUTPUT_DIR}")
else:
    print(" Folder model lokal sudah ada.")

## 3. Visualisasi & Verifikasi Dataset (Opsional)
Sebelum memulai ulang *training*, jalankan *cell* ini untuk memastikan dataset yang diekstrak ke lokal sudah terhitung dengan benar dan strukturnya siap digunakan.

In [ ]:
import pandas as pd
import os

master_classes = [
    "person", "bicycle", "car", "motorcycle", "bus", "pole",
    "tactile_paving_straight", "tactile_paving_turn", 
    "tactile_paving_3way", "tactile_paving_4way", "tactile_paving_stop",
    "stairs_up", "stairs_down", "crosswalk", "tree"
]

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test
    
    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total
    
    data_report.append({
        'ID': i, 
        'Kelas': cls_name, 
        'Train (Inst)': t_train, 
        'Valid (Inst)': t_valid, 
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-', 
    'Kelas': 'TOTAL KESELURUHAN', 
    'Train (Inst)': total_train, 
    'Valid (Inst)': total_valid, 
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


## 4. Memulai Ulang Training (Resume)
Sistem YOLO akan secara otomatis mendeteksi file `last.pt` beserta seluruh log masa lalunya (termasuk folder aslinya) yang berada di penyimpanan Runtime Colab, lalu melanjutkannya dari *epoch* terakhir mesin tersebut terputus.

In [ ]:
from ultralytics import YOLO

# Memanggil last.pt dari penyimpanan runtime LOKAL
model_path = f'{model_dir}/yolo11n_custom/weights/last.pt'

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(" Memulai Resume Training menggunakan memori lokal...")
    
    # Melanjutkan training. YOLO akan membaca results.csv yang ada di folder yang sama
    results = model.train(resume=True)
else:
    print(f" ERROR: File {model_path} tidak ditemukan! Pastikan ekstrak berhasil di cell sebelumnya.")

## 5. Simpan (Backup) Hasil Lanjutan Kembali ke Drive
Setelah proses *training* lanjutan selesai, kita WAJIB membungkus (ZIP) folder hasil kerja di Runtime Colab dan mengirimkannya kembali ke Google Drive agar bisa digunakan di masa depan, atau agar bisa diunduh untuk skripsi Anda.

In [ ]:
import os
import shutil

print(f"\nMenge-ZIP dan membackup seluruh hasil training ke {OUTPUT_DIR}...")
# Menyimpan langsung ke variabel OUTPUT_DIR yang sudah di-set di awal notebook
shutil.make_archive(f"{OUTPUT_DIR}/vnetra_training_results", 'zip', "/content/runs/detect/vnetra_training")

print(f"\nMenyalin file model (.pt) secara langsung (tanpa di-zip) ke {OUTPUT_DIR}...")
weights_dir = "/content/runs/detect/vnetra_training/yolo11n_custom/weights"
if os.path.exists(weights_dir):
    for pt_file in ["best.pt", "last.pt"]:
        src_pt = f"{weights_dir}/{pt_file}"
        dst_pt = f"{OUTPUT_DIR}/{pt_file}"
        if os.path.exists(src_pt):
            shutil.copy2(src_pt, dst_pt)
            print(f"✔️ Berhasil menyalin {pt_file}")
        else:
            print(f"⚠️ Peringatan: {pt_file} tidak ditemukan di {weights_dir}")

print(f"\n✅ BERHASIL! Seluruh grafik & log aman di ZIP, dan file Model bisa langsung diakses di:")
print(f"📁 {OUTPUT_DIR}/")
